In [1]:
"""Full-data LightGBM feature experiments for FreshRetailNet-50K.

Paste this entire file into one Kaggle notebook cell and run it after attaching
exactly one train.parquet and one eval.parquet below /kaggle/input.

Primary evaluation is fixed-origin 7-day-ahead. One-step LightGBM predictions
are fed back recursively and no realized sales or stockout state from the
forecast window is used. Secondary evaluation is rolling-origin one-day-ahead,
where a completed day's realized sales and stockout state become available
before forecasting the following day.

The weather-enhanced experiment uses validation/test weather as known-ahead
context. Treat it as an oracle upper bound unless genuine weather forecasts are
available at the forecast origin.
"""

from __future__ import annotations

import argparse
import gc
import json
import math
import os
import platform
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except ImportError as exc:  # pragma: no cover - Kaggle normally includes LightGBM
    raise ImportError(
        "LightGBM is required. In Kaggle, install it once with: !pip install lightgbm"
    ) from exc


TRAIN_START = "2024-03-28"
VALIDATION_TRAIN_END = "2024-06-18"
VALIDATION_START = "2024-06-19"
VALIDATION_END = "2024-06-25"
FINAL_TRAIN_END = "2024-06-25"
TEST_START = "2024-06-26"
TEST_END = "2024-07-02"

TRAIN_PATH: str | None = None
EVAL_PATH: str | None = None
OUTPUT_DIR = "/kaggle/working/freshretail-lightgbm-features"
RANDOM_SEED = 42

SERIES_KEYS = ["store_id", "product_id"]
CATEGORICAL_COLUMNS = [
    "city_id",
    "store_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "product_id",
]
CONTEXT_COLUMNS = ["discount", "holiday_flag", "activity_flag"]
WEATHER_COLUMNS = [
    "precpt",
    "avg_temperature",
    "avg_humidity",
    "avg_wind_level",
]
DYNAMIC_COLUMNS = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "rolling_mean_3",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_std_7",
    "past_stockout_count_7",
    "past_stockout_rate_7",
    "past_stockout_count_14",
    "past_stockout_rate_14",
]

OPERATIONAL_FEATURES = CATEGORICAL_COLUMNS + [
    "day_of_week",
    "is_weekend",
    "day_index",
    "holiday_flag",
    "activity_flag",
    "discount",
] + DYNAMIC_COLUMNS

FEATURE_SETS = {
    "operational": OPERATIONAL_FEATURES,
    "weather_enhanced": OPERATIONAL_FEATURES + WEATHER_COLUMNS,
}

REQUIRED_COLUMNS = list(
    dict.fromkeys(
        CATEGORICAL_COLUMNS
        + ["dt", "sale_amount", "stock_hour6_22_cnt"]
        + CONTEXT_COLUMNS
        + WEATHER_COLUMNS
    )
)

# Five predefined candidates. No early stopping is used because configuration
# selection must be based on leakage-free recursive fixed-origin WAPE.
LIGHTGBM_CONFIGS: dict[str, dict[str, Any]] = {
    "config_1_fast": {
        "n_estimators": 180,
        "learning_rate": 0.08,
        "num_leaves": 31,
        "min_child_samples": 120,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
    },
    "config_2_balanced": {
        "n_estimators": 350,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_child_samples": 100,
        "subsample": 0.85,
        "colsample_bytree": 0.90,
    },
    "config_3_deeper": {
        "n_estimators": 350,
        "learning_rate": 0.05,
        "num_leaves": 127,
        "max_depth": 12,
        "min_child_samples": 60,
        "subsample": 0.85,
        "colsample_bytree": 0.90,
    },
    "config_4_regularized": {
        "n_estimators": 450,
        "learning_rate": 0.04,
        "num_leaves": 63,
        "min_child_samples": 200,
        "reg_alpha": 0.50,
        "reg_lambda": 1.00,
        "subsample": 0.80,
        "colsample_bytree": 0.85,
    },
    "config_5_low_lr": {
        "n_estimators": 650,
        "learning_rate": 0.025,
        "num_leaves": 63,
        "min_child_samples": 100,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
    },
}

COMMON_MODEL_PARAMS: dict[str, Any] = {
    "objective": "regression_l1",
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
    "verbosity": -1,
    "max_bin": 127,
    "subsample_freq": 1,
    "force_col_wise": True,
}


def discover_parquet(filename: str, explicit: str | None = None) -> Path:
    """Find exactly one named parquet file, preferring an explicit path."""
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"Explicit path does not exist: {path}")
        return path

    env_name = f"FRESHRETAIL_{Path(filename).stem.upper()}_PATH"
    if os.environ.get(env_name):
        path = Path(os.environ[env_name]).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
        return path

    root = Path("/kaggle/input")
    matches = sorted({path.resolve() for path in root.rglob(filename) if path.is_file()}, key=str)
    if not matches:
        raise FileNotFoundError(f"No {filename} was found below /kaggle/input")
    if len(matches) != 1:
        choices = "\n".join(f"- {path}" for path in matches)
        raise RuntimeError(f"Expected exactly one {filename}; found {len(matches)}:\n{choices}")
    return matches[0]


def _downcast_frame(frame: pd.DataFrame) -> pd.DataFrame:
    for column in CATEGORICAL_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], downcast="integer")
    frame["sale_amount"] = frame["sale_amount"].astype(np.float32)
    frame["stock_hour6_22_cnt"] = pd.to_numeric(
        frame["stock_hour6_22_cnt"], downcast="integer"
    )
    for column in ["holiday_flag", "activity_flag"]:
        frame[column] = pd.to_numeric(frame[column], downcast="integer")
    for column in ["discount"] + WEATHER_COLUMNS:
        frame[column] = frame[column].astype(np.float32)
    return frame


def _validate_panel(
    frame: pd.DataFrame,
    expected_start: str,
    expected_end: str,
    expected_days: int,
    label: str,
) -> tuple[pd.DatetimeIndex, pd.DataFrame]:
    if frame[REQUIRED_COLUMNS].isna().any().any():
        missing = frame[REQUIRED_COLUMNS].isna().sum()
        raise ValueError(f"{label} contains missing required values:\n{missing[missing > 0]}")
    if frame.duplicated(SERIES_KEYS + ["dt"]).any():
        raise ValueError(f"{label} contains duplicate store-product-date keys")

    dates = pd.DatetimeIndex(frame["dt"].drop_duplicates().sort_values())
    expected = pd.date_range(expected_start, expected_end, freq="D")
    if len(dates) != expected_days or not dates.equals(expected):
        raise ValueError(
            f"{label} must cover {expected_start}..{expected_end}; found "
            f"{dates.min().date()}..{dates.max().date()} ({len(dates)} days)"
        )

    counts = frame.groupby(SERIES_KEYS, sort=False, observed=True).size()
    if len(counts) != 50_000 or counts.nunique() != 1 or int(counts.iloc[0]) != expected_days:
        raise ValueError(
            f"{label} must be a complete 50,000 x {expected_days} panel; "
            f"found {len(counts):,} series and {len(frame):,} rows"
        )
    if len(frame) != 50_000 * expected_days:
        raise ValueError(f"Unexpected {label} row count: {len(frame):,}")

    series = frame.loc[::expected_days, SERIES_KEYS].reset_index(drop=True)
    return dates, series


def load_data(train_path: Path, eval_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    """Load only required columns, validate the two complete panels, and downcast."""
    print("Loading required columns from train.parquet ...", flush=True)
    train = pd.read_parquet(train_path, columns=REQUIRED_COLUMNS)
    print("Loading required columns from eval.parquet ...", flush=True)
    eval_frame = pd.read_parquet(eval_path, columns=REQUIRED_COLUMNS)

    for frame in (train, eval_frame):
        frame["dt"] = pd.to_datetime(frame["dt"], format="%Y-%m-%d")
    train = train.sort_values(SERIES_KEYS + ["dt"], kind="stable").reset_index(drop=True)
    eval_frame = eval_frame.sort_values(SERIES_KEYS + ["dt"], kind="stable").reset_index(drop=True)
    train = _downcast_frame(train)
    eval_frame = _downcast_frame(eval_frame)

    train_dates, train_series = _validate_panel(
        train, TRAIN_START, FINAL_TRAIN_END, 90, "train.parquet"
    )
    eval_dates, eval_series = _validate_panel(
        eval_frame, TEST_START, TEST_END, 7, "eval.parquet"
    )
    if not np.array_equal(train_series.to_numpy(), eval_series.to_numpy()):
        raise ValueError("Train and eval store-product series/order do not match")

    category_levels: dict[str, list[int]] = {}
    for column in CATEGORICAL_COLUMNS:
        levels = np.sort(train[column].unique())
        unseen = np.setdiff1d(eval_frame[column].unique(), levels)
        if unseen.size:
            raise ValueError(f"Eval contains unseen {column} values: {unseen[:10].tolist()}")
        category_levels[column] = [int(value) for value in levels]
        train[column] = pd.Categorical(train[column], categories=levels)
        eval_frame[column] = pd.Categorical(eval_frame[column], categories=levels)

    metadata = {
        "train_dates": train_dates,
        "eval_dates": eval_dates,
        "series": train_series,
        "category_levels": category_levels,
    }
    return train, eval_frame, metadata


def add_calendar_features(frame: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Add deterministic calendar features known at every forecast origin."""
    day_of_week = frame["dt"].dt.dayofweek
    frame["day_of_week"] = day_of_week.astype(np.int8)
    frame["is_weekend"] = (day_of_week >= 5).astype(np.int8)
    frame["day_index"] = (frame["dt"] - origin).dt.days.astype(np.int16)
    return frame


def _shift_matrix(values: np.ndarray, lag: int) -> np.ndarray:
    shifted = np.full(values.shape, np.nan, dtype=np.float32)
    shifted[:, lag:] = values[:, :-lag]
    return shifted


def _past_window_sum(values: np.ndarray, window: int) -> np.ndarray:
    """At column t return sum(values[t-window:t]); current t is excluded."""
    n_series, n_dates = values.shape
    result = np.full((n_series, n_dates), np.nan, dtype=np.float32)
    cumulative = np.empty((n_series, n_dates + 1), dtype=np.float64)
    cumulative[:, 0] = 0.0
    np.cumsum(values, axis=1, dtype=np.float64, out=cumulative[:, 1:])
    result[:, window:] = (
        cumulative[:, window:n_dates] - cumulative[:, : n_dates - window]
    ).astype(np.float32)
    return result


def create_lag_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create leakage-safe lags/rollings; every window ends at t-1."""
    n_dates = frame["dt"].nunique()
    n_series = len(frame) // n_dates
    sales = frame["sale_amount"].to_numpy(dtype=np.float32, copy=False).reshape(n_series, n_dates)
    stockout_day = (
        frame["stock_hour6_22_cnt"].to_numpy(copy=False).reshape(n_series, n_dates) > 0
    ).astype(np.float32)

    for lag in (1, 2, 3, 7, 14):
        frame[f"lag_{lag}"] = _shift_matrix(sales, lag).reshape(-1)

    for window in (3, 7, 14):
        past_sum = _past_window_sum(sales, window)
        frame[f"rolling_mean_{window}"] = (past_sum / np.float32(window)).reshape(-1)

    # A sliding view avoids numerical cancellation from E[x^2] - E[x]^2 and
    # exactly matches the dynamic np.std calculation used during forecasting.
    rolling_std_7 = np.full(sales.shape, np.nan, dtype=np.float32)
    sales_windows_7 = np.lib.stride_tricks.sliding_window_view(sales, 7, axis=1)
    rolling_std_7[:, 7:] = sales_windows_7[:, :-1].std(axis=2, dtype=np.float32)
    frame["rolling_std_7"] = rolling_std_7.reshape(-1)

    for window in (7, 14):
        count = _past_window_sum(stockout_day, window)
        frame[f"past_stockout_count_{window}"] = count.reshape(-1)
        frame[f"past_stockout_rate_{window}"] = (count / np.float32(window)).reshape(-1)
    return frame


def train_lightgbm(
    training_frame: pd.DataFrame,
    feature_columns: list[str],
    config_name: str,
    config: dict[str, Any],
    training_end: str,
) -> LGBMRegressor:
    """Fit one predefined LightGBM configuration on all eligible training rows."""
    # lag_14 is the last feature to become available, and source columns were
    # already checked for missing values. This mask avoids materializing a large
    # temporary boolean frame for all 26/30 features on every candidate run.
    valid_rows = training_frame["lag_14"].notna() & (
        training_frame["dt"] <= pd.Timestamp(training_end)
    )
    x_train = training_frame.loc[valid_rows, feature_columns]
    y_train = training_frame.loc[valid_rows, "sale_amount"].to_numpy(
        dtype=np.float32, copy=False
    )
    print(
        f"    fitting {config_name}: {len(x_train):,} rows, {len(feature_columns)} features",
        flush=True,
    )
    model = LGBMRegressor(**COMMON_MODEL_PARAMS, **config)
    model.fit(
        x_train,
        y_train,
        categorical_feature=CATEGORICAL_COLUMNS,
        callbacks=[lgb.log_evaluation(period=0)],
    )
    del x_train, y_train, valid_rows
    gc.collect()
    return model


def _dynamic_features(
    sales_history: np.ndarray,
    stockout_history: np.ndarray,
) -> dict[str, np.ndarray]:
    if sales_history.shape[1] < 14 or stockout_history.shape[1] < 14:
        raise ValueError("At least 14 pre-origin days are required")
    stockout_day = stockout_history > 0
    return {
        "lag_1": sales_history[:, -1],
        "lag_2": sales_history[:, -2],
        "lag_3": sales_history[:, -3],
        "lag_7": sales_history[:, -7],
        "lag_14": sales_history[:, -14],
        "rolling_mean_3": sales_history[:, -3:].mean(axis=1, dtype=np.float32),
        "rolling_mean_7": sales_history[:, -7:].mean(axis=1, dtype=np.float32),
        "rolling_mean_14": sales_history[:, -14:].mean(axis=1, dtype=np.float32),
        "rolling_std_7": sales_history[:, -7:].std(axis=1, dtype=np.float32),
        "past_stockout_count_7": stockout_day[:, -7:].sum(axis=1).astype(np.float32),
        "past_stockout_rate_7": stockout_day[:, -7:].mean(axis=1).astype(np.float32),
        "past_stockout_count_14": stockout_day[:, -14:].sum(axis=1).astype(np.float32),
        "past_stockout_rate_14": stockout_day[:, -14:].mean(axis=1).astype(np.float32),
    }


def _forecast_day_features(
    forecast_frame: pd.DataFrame,
    forecast_date: pd.Timestamp,
    sales_history: np.ndarray,
    stockout_history: np.ndarray,
    feature_columns: list[str],
    n_series: int,
) -> pd.DataFrame:
    # Eval has no precomputed dynamic columns by design: they must be generated
    # from origin history rather than from actual future values.
    base_columns = [column for column in feature_columns if column not in DYNAMIC_COLUMNS]
    rows = forecast_frame.loc[
        forecast_frame["dt"] == forecast_date, base_columns
    ].copy()
    if len(rows) != n_series:
        raise ValueError(f"Expected {n_series:,} rows on {forecast_date.date()}, found {len(rows):,}")
    dynamic = _dynamic_features(sales_history, stockout_history)
    for column, values in dynamic.items():
        rows[column] = values
    return rows[feature_columns]


def forecast_fixed_origin_lightgbm(
    model: LGBMRegressor,
    history_sales: np.ndarray,
    history_stockout: np.ndarray,
    forecast_frame: pd.DataFrame,
    forecast_dates: pd.DatetimeIndex,
    feature_columns: list[str],
) -> np.ndarray:
    """Recursive seven-day forecast with one cutoff and no future actual state."""
    n_series, history_days = history_sales.shape
    horizon = len(forecast_dates)
    sales_extended = np.empty((n_series, history_days + horizon), dtype=np.float32)
    stockout_extended = np.empty((n_series, history_days + horizon), dtype=np.int8)
    sales_extended[:, :history_days] = history_sales
    stockout_extended[:, :history_days] = history_stockout
    predictions = np.empty((n_series, horizon), dtype=np.float32)

    for step, forecast_date in enumerate(forecast_dates):
        features = _forecast_day_features(
            forecast_frame,
            forecast_date,
            sales_extended[:, : history_days + step],
            stockout_extended[:, : history_days + step],
            feature_columns,
            n_series,
        )
        next_prediction = np.maximum(model.predict(features), 0.0).astype(np.float32)
        predictions[:, step] = next_prediction
        sales_extended[:, history_days + step] = next_prediction
        # Stockout inside the fixed-origin future is unobserved and is never read
        # from validation/eval. Zero is the explicit no-observed-stockout assumption.
        stockout_extended[:, history_days + step] = 0
        del features
    return predictions


def forecast_rolling_origin_lightgbm(
    model: LGBMRegressor,
    history_sales: np.ndarray,
    history_stockout: np.ndarray,
    forecast_frame: pd.DataFrame,
    forecast_dates: pd.DatetimeIndex,
    actual_sales: np.ndarray,
    actual_stockout: np.ndarray,
    feature_columns: list[str],
) -> np.ndarray:
    """One-day-ahead forecasts updated with each completed day's actual state."""
    n_series, history_days = history_sales.shape
    horizon = len(forecast_dates)
    sales_extended = np.empty((n_series, history_days + horizon), dtype=np.float32)
    stockout_extended = np.empty((n_series, history_days + horizon), dtype=np.int8)
    sales_extended[:, :history_days] = history_sales
    stockout_extended[:, :history_days] = history_stockout
    predictions = np.empty((n_series, horizon), dtype=np.float32)

    for step, forecast_date in enumerate(forecast_dates):
        features = _forecast_day_features(
            forecast_frame,
            forecast_date,
            sales_extended[:, : history_days + step],
            stockout_extended[:, : history_days + step],
            feature_columns,
            n_series,
        )
        predictions[:, step] = np.maximum(model.predict(features), 0.0).astype(np.float32)
        sales_extended[:, history_days + step] = actual_sales[:, step]
        stockout_extended[:, history_days + step] = actual_stockout[:, step]
        del features
    return predictions


def metrics(
    actual: np.ndarray,
    prediction: np.ndarray,
    eligible: np.ndarray,
) -> dict[str, float | int]:
    y = actual[eligible].astype(np.float64, copy=False)
    y_hat = np.maximum(prediction[eligible].astype(np.float64, copy=False), 0.0)
    if y.size == 0:
        return {name: float("nan") for name in [
            "mae", "rmse", "wape", "wpe", "underestimation_rate", "r2",
            "actual_sum", "prediction_sum"
        ]} | {"n": 0}
    error = y_hat - y
    absolute_error = np.abs(error)
    denominator = np.abs(y).sum()
    squared_error = np.square(error).sum()
    centered = np.square(y - y.mean()).sum()
    return {
        "n": int(y.size),
        "mae": float(absolute_error.mean()),
        "rmse": float(math.sqrt(squared_error / y.size)),
        "wape": float(absolute_error.sum() / denominator) if denominator else float("nan"),
        "wpe": float(error.sum() / denominator) if denominator else float("nan"),
        "underestimation_rate": float(np.mean(y_hat < y)),
        "r2": float(1.0 - squared_error / centered) if centered else float("nan"),
        "actual_sum": float(y.sum()),
        "prediction_sum": float(y_hat.sum()),
    }


def horizon_metrics(
    feature_set: str,
    config_name: str,
    protocol: str,
    split: str,
    actual: np.ndarray,
    prediction: np.ndarray,
    eligible: np.ndarray,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for horizon_index in range(actual.shape[1]):
        rows.append(
            {
                "model": "lightgbm",
                "feature_set": feature_set,
                "config": config_name,
                "protocol": protocol,
                "split": split,
                "horizon": horizon_index + 1,
            }
            | metrics(
                actual[:, horizon_index],
                prediction[:, horizon_index],
                eligible[:, horizon_index],
            )
        )
    return rows


def cumulative_metrics(
    actual: np.ndarray,
    prediction: np.ndarray,
    eligible: np.ndarray,
) -> dict[str, float | int]:
    """Score seven-day totals only for series uncensored on all seven days."""
    fully_observed_series = eligible.all(axis=1)
    return metrics(
        actual.sum(axis=1),
        prediction.sum(axis=1),
        fully_observed_series,
    )


def _panel_arrays(frame: pd.DataFrame, n_dates: int) -> tuple[np.ndarray, np.ndarray]:
    n_series = len(frame) // n_dates
    sales = frame["sale_amount"].to_numpy(dtype=np.float32, copy=False).reshape(n_series, n_dates)
    stockout = frame["stock_hour6_22_cnt"].to_numpy(copy=False).reshape(n_series, n_dates)
    return sales, stockout.astype(np.int8, copy=False)


def _evaluate_predictions(
    feature_set: str,
    config_name: str,
    split: str,
    protocol_predictions: dict[str, np.ndarray],
    actual: np.ndarray,
    stockout: np.ndarray,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]]]:
    eligible = stockout == 0
    overall_rows: list[dict[str, Any]] = []
    horizon_rows: list[dict[str, Any]] = []
    cumulative_rows: list[dict[str, Any]] = []
    for protocol, prediction in protocol_predictions.items():
        overall_rows.append(
            {
                "model": "lightgbm",
                "feature_set": feature_set,
                "config": config_name,
                "protocol": protocol,
                "target": "observed_sales",
                "split": split,
            }
            | metrics(actual, prediction, eligible)
        )
        horizon_rows.extend(
            horizon_metrics(
                feature_set,
                config_name,
                protocol,
                split,
                actual,
                prediction,
                eligible,
            )
        )
        cumulative_rows.append(
            {
                "model": "lightgbm",
                "feature_set": feature_set,
                "config": config_name,
                "protocol": protocol,
                "target": "observed_sales",
                "split": split,
            }
            | cumulative_metrics(actual, prediction, eligible)
        )
    return overall_rows, horizon_rows, cumulative_rows


def _print_result(prefix: str, rows: list[dict[str, Any]]) -> None:
    for row in rows:
        print(
            f"{prefix} {row['protocol']:22s} WAPE={row['wape']:.4f} "
            f"WPE={row['wpe']:.4f} R2={row['r2']:.4f}",
            flush=True,
        )


def main() -> None:
    parser = argparse.ArgumentParser(description="FreshRetailNet-50K LightGBM feature experiment")
    parser.add_argument("--train-path", default=TRAIN_PATH)
    parser.add_argument("--eval-path", default=EVAL_PATH)
    parser.add_argument("--output-dir", default=OUTPUT_DIR)
    args, _unknown = parser.parse_known_args()

    started = time.perf_counter()
    output_dir = Path(args.output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    train_path = discover_parquet("train.parquet", args.train_path)
    eval_path = discover_parquet("eval.parquet", args.eval_path)
    print(f"Train: {train_path}", flush=True)
    print(f"Eval:  {eval_path}", flush=True)

    train, eval_frame, metadata = load_data(train_path, eval_path)
    origin = pd.Timestamp(TRAIN_START)
    train = add_calendar_features(train, origin)
    eval_frame = add_calendar_features(eval_frame, origin)
    print("Creating leakage-safe lag, rolling, and past-stockout features ...", flush=True)
    train = create_lag_features(train)

    train_dates: pd.DatetimeIndex = metadata["train_dates"]
    eval_dates: pd.DatetimeIndex = metadata["eval_dates"]
    series: pd.DataFrame = metadata["series"]
    train_sales, train_stockout = _panel_arrays(train, len(train_dates))
    eval_sales, eval_stockout = _panel_arrays(eval_frame, len(eval_dates))
    validation_dates = pd.date_range(VALIDATION_START, VALIDATION_END, freq="D")
    validation_start_index = int(np.flatnonzero(train_dates == pd.Timestamp(VALIDATION_START))[0])
    validation_actual = train_sales[:, validation_start_index:]
    validation_stockout = train_stockout[:, validation_start_index:]
    validation_history_sales = train_sales[:, :validation_start_index]
    validation_history_stockout = train_stockout[:, :validation_start_index]
    validation_context = train.loc[
        (train["dt"] >= pd.Timestamp(VALIDATION_START))
        & (train["dt"] <= pd.Timestamp(VALIDATION_END))
    ]

    validation_rows: list[dict[str, Any]] = []
    validation_horizon_rows: list[dict[str, Any]] = []
    validation_cumulative_rows: list[dict[str, Any]] = []

    for feature_set, feature_columns in FEATURE_SETS.items():
        print(f"\nFEATURE SET: {feature_set}", flush=True)
        for config_name, config in LIGHTGBM_CONFIGS.items():
            model = train_lightgbm(
                train,
                feature_columns,
                config_name,
                config,
                VALIDATION_TRAIN_END,
            )
            fixed_prediction = forecast_fixed_origin_lightgbm(
                model,
                validation_history_sales,
                validation_history_stockout,
                validation_context,
                validation_dates,
                feature_columns,
            )
            rolling_prediction = forecast_rolling_origin_lightgbm(
                model,
                validation_history_sales,
                validation_history_stockout,
                validation_context,
                validation_dates,
                validation_actual,
                validation_stockout,
                feature_columns,
            )
            overall, by_horizon, cumulative = _evaluate_predictions(
                feature_set,
                config_name,
                "validation",
                {
                    "fixed_origin_7_day": fixed_prediction,
                    "rolling_origin_1_day": rolling_prediction,
                },
                validation_actual,
                validation_stockout,
            )
            validation_rows.extend(overall)
            validation_horizon_rows.extend(by_horizon)
            validation_cumulative_rows.extend(cumulative)
            _print_result(f"  {config_name:22s}", overall)
            del model, fixed_prediction, rolling_prediction
            gc.collect()

    validation_results = pd.DataFrame(validation_rows)
    validation_results["absolute_wpe"] = validation_results["wpe"].abs()
    validation_results = validation_results.sort_values(
        ["protocol", "feature_set", "wape", "absolute_wpe", "underestimation_rate", "mae"],
        kind="stable",
    ).reset_index(drop=True)
    validation_results["feature_set_rank"] = validation_results.groupby(
        ["protocol", "feature_set"], sort=False
    ).cumcount() + 1
    validation_results.to_csv(output_dir / "lightgbm_validation_full_data.csv", index=False)
    pd.DataFrame(validation_horizon_rows).to_csv(
        output_dir / "lightgbm_validation_by_horizon.csv", index=False
    )
    pd.DataFrame(validation_cumulative_rows).to_csv(
        output_dir / "lightgbm_validation_cumulative_7_day.csv", index=False
    )

    primary = validation_results.loc[
        validation_results["protocol"] == "fixed_origin_7_day"
    ]
    best = primary.loc[primary["feature_set_rank"] == 1].copy()
    best_configs = {
        row.feature_set: row.config for row in best.itertuples(index=False)
    }
    print("\nLOCKED WINNERS FROM FIXED-ORIGIN VALIDATION:", flush=True)
    print(best[["feature_set", "config", "wape", "wpe", "r2"]].to_string(index=False), flush=True)

    final_rows: list[dict[str, Any]] = []
    final_horizon_rows: list[dict[str, Any]] = []
    final_cumulative_rows: list[dict[str, Any]] = []
    final_predictions: dict[tuple[str, str], np.ndarray] = {}
    for feature_set, feature_columns in FEATURE_SETS.items():
        config_name = best_configs[feature_set]
        print(f"\nFINAL REFIT: {feature_set} / {config_name}", flush=True)
        model = train_lightgbm(
            train,
            feature_columns,
            config_name,
            LIGHTGBM_CONFIGS[config_name],
            FINAL_TRAIN_END,
        )
        fixed_prediction = forecast_fixed_origin_lightgbm(
            model,
            train_sales,
            train_stockout,
            eval_frame,
            eval_dates,
            feature_columns,
        )
        rolling_prediction = forecast_rolling_origin_lightgbm(
            model,
            train_sales,
            train_stockout,
            eval_frame,
            eval_dates,
            eval_sales,
            eval_stockout,
            feature_columns,
        )
        final_predictions[(feature_set, "fixed_origin_7_day")] = fixed_prediction
        final_predictions[(feature_set, "rolling_origin_1_day")] = rolling_prediction
        overall, by_horizon, cumulative = _evaluate_predictions(
            feature_set,
            config_name,
            "test",
            {
                "fixed_origin_7_day": fixed_prediction,
                "rolling_origin_1_day": rolling_prediction,
            },
            eval_sales,
            eval_stockout,
        )
        final_rows.extend(overall)
        final_horizon_rows.extend(by_horizon)
        final_cumulative_rows.extend(cumulative)
        _print_result("  final", overall)
        del model
        gc.collect()

    final_results = pd.DataFrame(final_rows).sort_values(
        ["protocol", "wape"], kind="stable"
    ).reset_index(drop=True)
    final_results.to_csv(output_dir / "lightgbm_final_test.csv", index=False)
    final_horizon_frame = pd.DataFrame(final_horizon_rows)
    final_horizon_frame.to_csv(output_dir / "lightgbm_final_test_by_horizon.csv", index=False)
    pd.DataFrame(final_cumulative_rows).to_csv(
        output_dir / "lightgbm_final_test_cumulative_7_day.csv", index=False
    )

    error_growth = final_horizon_frame.pivot_table(
        index=["model", "feature_set", "config", "protocol"],
        columns="horizon",
        values="wape",
    ).reset_index()
    error_growth["wape_growth_h7_minus_h1"] = error_growth[7] - error_growth[1]
    error_growth = error_growth.rename(columns={1: "wape_h1", 7: "wape_h7"})
    error_growth.to_csv(output_dir / "lightgbm_final_test_error_growth.csv", index=False)

    prediction_frame = pd.DataFrame(
        {
            "store_id": np.repeat(series["store_id"].to_numpy(dtype=np.int32), len(eval_dates)),
            "product_id": np.repeat(series["product_id"].to_numpy(dtype=np.int32), len(eval_dates)),
            "dt": np.tile(eval_dates.to_numpy(), len(series)),
            "actual_sale_amount": eval_sales.reshape(-1),
            "stockout_hours": eval_stockout.reshape(-1),
            "eligible_non_stockout": (eval_stockout == 0).reshape(-1),
            "prediction_operational_fixed": final_predictions[
                ("operational", "fixed_origin_7_day")
            ].reshape(-1),
            "prediction_operational_rolling": final_predictions[
                ("operational", "rolling_origin_1_day")
            ].reshape(-1),
            "prediction_weather_fixed": final_predictions[
                ("weather_enhanced", "fixed_origin_7_day")
            ].reshape(-1),
            "prediction_weather_rolling": final_predictions[
                ("weather_enhanced", "rolling_origin_1_day")
            ].reshape(-1),
        }
    )
    prediction_frame.to_parquet(
        output_dir / "lightgbm_best_model_predictions.parquet", index=False
    )

    manifest = {
        "experiment": "lightgbm_observed_sales_operational_vs_weather_enhanced",
        "full_data": True,
        "train_path": str(train_path),
        "eval_path": str(eval_path),
        "date_ranges": {
            "validation_train": [TRAIN_START, VALIDATION_TRAIN_END],
            "validation": [VALIDATION_START, VALIDATION_END],
            "final_refit": [TRAIN_START, FINAL_TRAIN_END],
            "final_test": [TEST_START, TEST_END],
        },
        "rows": {"train": int(len(train)), "eval": int(len(eval_frame))},
        "series": int(len(series)),
        "target": "observed_sales",
        "feature_sets": FEATURE_SETS,
        "weather_setting": (
            "Weather-enhanced validation/test covariates are treated as known-ahead/oracle "
            "features unless genuine forecast weather is supplied."
        ),
        "configs": LIGHTGBM_CONFIGS,
        "common_model_params": COMMON_MODEL_PARAMS,
        "best_config_per_feature_set": best_configs,
        "primary_protocol": "fixed_origin_7_day",
        "secondary_protocol": "rolling_origin_1_day",
        "selection_rule": "lowest fixed-origin validation WAPE per feature set",
        "rolling_result_note": (
            "Rolling results use fixed-origin-selected configurations; no rolling-specific retuning."
        ),
        "anti_leakage": {
            "lag_and_rolling_rule": "all historical windows end at t-1",
            "fixed_sales_update": "recursive predictions",
            "fixed_future_stockout_update": "zero/no-observed-stockout assumption",
            "rolling_update": "completed prior-day actual sales and stockout state",
            "same_day_sales_feature": False,
            "same_day_stockout_feature": False,
        },
        "stockout_feature_definition": (
            "past_stockout_count_k counts prior days with stock_hour6_22_cnt > 0; "
            "past_stockout_rate_k divides that count by k"
        ),
        "rolling_std_ddof": 0,
        "evaluation_eligibility": "stock_hour6_22_cnt == 0",
        "cumulative_eligibility": "series has no stockout on all seven forecast days",
        "validation_results": validation_results.to_dict(orient="records"),
        "final_test_results": final_results.to_dict(orient="records"),
        "runtime_seconds": time.perf_counter() - started,
        "python": platform.python_version(),
        "lightgbm": lgb.__version__,
    }
    (output_dir / "run_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )

    print("\nFINAL TEST — fixed-origin-selected winners under both protocols:", flush=True)
    print(final_results.to_string(index=False), flush=True)
    print(f"\nSaved outputs to: {output_dir}", flush=True)
    print(f"Runtime: {(time.perf_counter() - started) / 60:.1f} minutes", flush=True)


if __name__ == "__main__":
    main()



Train: /kaggle/input/datasets/takedo06/datafresh/train.parquet
Eval:  /kaggle/input/datasets/takedo06/datafresh/eval.parquet
Loading required columns from train.parquet ...
Loading required columns from eval.parquet ...
Creating leakage-safe lag, rolling, and past-stockout features ...

FEATURE SET: operational
    fitting config_1_fast: 3,450,000 rows, 26 features
  config_1_fast          fixed_origin_7_day     WAPE=0.3055 WPE=-0.1195 R2=0.8801
  config_1_fast          rolling_origin_1_day   WAPE=0.2895 WPE=-0.0829 R2=0.8950
    fitting config_2_balanced: 3,450,000 rows, 26 features
  config_2_balanced      fixed_origin_7_day     WAPE=0.3011 WPE=-0.1086 R2=0.8865
  config_2_balanced      rolling_origin_1_day   WAPE=0.2854 WPE=-0.0757 R2=0.9018
    fitting config_3_deeper: 3,450,000 rows, 26 features
  config_3_deeper        fixed_origin_7_day     WAPE=0.2994 WPE=-0.1102 R2=0.8894
  config_3_deeper        rolling_origin_1_day   WAPE=0.2838 WPE=-0.0773 R2=0.9048
    fitting config_4_reg